In [0]:
# 07 — Data Quality Checks
# Validates data integrity at every layer of the pipeline.
# These checks run AFTER the pipeline completes to verify
# that data is clean, complete, and realistic.
# In production this would block downstream processing
# if any check fails — preventing bad data from reaching
# compliance officers or regulators.

from pyspark.sql.functions import col, count, sum as spark_sum, isnan, isnull

BRONZE_TABLE = "aml_pipeline.transactions.bronze_transactions"
SILVER_TABLE = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE   = "aml_pipeline.transactions.gold_sar_reports"

passed = 0
failed = 0
results = []

def check(name, condition, detail=""):
    global passed, failed
    status = "PASS" if condition else "FAIL"
    if condition:
        passed += 1
    else:
        failed += 1
    results.append((name, status, detail))
    print(f"  [{status}] {name} {('— ' + detail) if detail else ''}")

# ── BRONZE LAYER CHECKS ─────────────────────────────────────
print("BRONZE LAYER CHECKS")
print("-" * 50)

bronze = spark.table(BRONZE_TABLE)
bronze_count = bronze.count()

# Check 1: Record count
check(
    "Record count is 100,000",
    bronze_count == 100_000,
    f"Got {bronze_count:,}"
)

# Check 2: No null transaction IDs
null_ids = bronze.filter(col("transaction_id").isNull()).count()
check(
    "No null transaction_id",
    null_ids == 0,
    f"{null_ids} nulls found"
)

# Check 3: No duplicate transaction IDs
distinct_ids = bronze.select("transaction_id").distinct().count()
check(
    "No duplicate transaction_id",
    distinct_ids == bronze_count,
    f"{bronze_count - distinct_ids} duplicates"
)

# Check 4: All required fields present
required_fields = ["transaction_id", "sender_name", "receiver_name",
                   "amount", "currency", "sender_country"]
for field in required_fields:
    null_count = bronze.filter(col(field).isNull()).count()
    check(
        f"Field '{field}' has no nulls",
        null_count == 0,
        f"{null_count} nulls"
    )

# Check 5: Amount is positive
neg_amounts = bronze.filter(col("amount") <= 0).count()
check(
    "All amounts are positive",
    neg_amounts == 0,
    f"{neg_amounts} non-positive amounts"
)

# Check 6: Ingestion timestamp exists
null_ts = bronze.filter(col("ingestion_timestamp").isNull()).count()
check(
    "All records have ingestion_timestamp",
    null_ts == 0,
    f"{null_ts} missing timestamps"
)

# ── SILVER LAYER CHECKS ─────────────────────────────────────
print(f"\nSILVER LAYER CHECKS")
print("-" * 50)

silver = spark.table(SILVER_TABLE)
silver_count = silver.count()

# Check 7: Record count matches Bronze
check(
    "Silver count matches Bronze",
    silver_count == bronze_count,
    f"Bronze={bronze_count:,}, Silver={silver_count:,}"
)

# Check 8: All records have amount_usd
null_usd = silver.filter(col("amount_usd").isNull()).count()
check(
    "All records have amount_usd",
    null_usd == 0,
    f"{null_usd} missing USD amounts"
)

# Check 9: amount_usd is positive
neg_usd = silver.filter(col("amount_usd") <= 0).count()
check(
    "All USD amounts are positive",
    neg_usd == 0,
    f"{neg_usd} non-positive"
)

# Check 10: Sanctions status populated
null_sanctions = silver.filter(col("sender_sanctions_status").isNull()).count()
check(
    "All records have sanctions status",
    null_sanctions == 0,
    f"{null_sanctions} missing"
)

# Check 11: Travel rule status populated
null_tr = silver.filter(col("travel_rule_status").isNull()).count()
check(
    "All records have travel rule status",
    null_tr == 0,
    f"{null_tr} missing"
)

# Check 12: Flag rate is realistic (1-10%)
flagged = silver.filter(col("is_flagged") == True).count()
flag_rate = flagged / silver_count * 100
check(
    "Flag rate is realistic (1-10%)",
    1.0 <= flag_rate <= 10.0,
    f"{round(flag_rate, 1)}%"
)

# Check 13: is_flagged column has no nulls
null_flagged = silver.filter(col("is_flagged").isNull()).count()
check(
    "No null is_flagged values",
    null_flagged == 0,
    f"{null_flagged} nulls"
)

# ── GOLD LAYER CHECKS ───────────────────────────────────────
print(f"\nGOLD LAYER CHECKS")
print("-" * 50)

gold = spark.table(GOLD_TABLE)
gold_count = gold.count()

# Check 14: Gold count matches flagged Silver count
check(
    "Gold count matches Silver flagged count",
    gold_count == flagged,
    f"Gold={gold_count:,}, Silver flagged={flagged:,}"
)

# Check 15: All Gold records have SAR reference
null_sar = gold.filter(col("sar_reference").isNull()).count()
check(
    "All SARs have reference number",
    null_sar == 0,
    f"{null_sar} missing"
)

# Check 16: All Gold records have severity
null_severity = gold.filter(col("sar_severity").isNull()).count()
check(
    "All SARs have severity level",
    null_severity == 0,
    f"{null_severity} missing"
)

# Check 17: All records are PENDING_REVIEW
non_pending = gold.filter(col("report_status") != "PENDING_REVIEW").count()
check(
    "All SARs are PENDING_REVIEW",
    non_pending == 0,
    f"{non_pending} not pending"
)

# Check 18: No duplicate SAR references
distinct_sars = gold.select("sar_reference").distinct().count()
check(
    "No duplicate SAR references",
    distinct_sars == gold_count,
    f"{gold_count - distinct_sars} duplicates"
)

# ── FINAL SUMMARY ────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"  DATA QUALITY REPORT")
print(f"{'='*50}")
print(f"  Total checks : {passed + failed}")
print(f"  Passed       : {passed}")
print(f"  Failed       : {failed}")
print(f"  Status       : {'ALL CHECKS PASSED' if failed == 0 else 'CHECKS FAILED — INVESTIGATE'}")
print(f"{'='*50}")

if failed > 0:
    print(f"\n  Failed checks:")
    for name, status, detail in results:
        if status == "FAIL":
            print(f"    - {name}: {detail}")

BRONZE LAYER CHECKS
--------------------------------------------------
  [PASS] Record count is 100,000 — Got 100,000
  [PASS] No null transaction_id — 0 nulls found
  [PASS] No duplicate transaction_id — 0 duplicates
  [PASS] Field 'transaction_id' has no nulls — 0 nulls
  [PASS] Field 'sender_name' has no nulls — 0 nulls
  [PASS] Field 'receiver_name' has no nulls — 0 nulls
  [PASS] Field 'amount' has no nulls — 0 nulls
  [PASS] Field 'currency' has no nulls — 0 nulls
  [PASS] Field 'sender_country' has no nulls — 0 nulls
  [PASS] All amounts are positive — 0 non-positive amounts
  [PASS] All records have ingestion_timestamp — 0 missing timestamps

SILVER LAYER CHECKS
--------------------------------------------------
  [PASS] Silver count matches Bronze — Bronze=100,000, Silver=100,000
  [PASS] All records have amount_usd — 0 missing USD amounts
  [PASS] All USD amounts are positive — 0 non-positive
  [PASS] All records have sanctions status — 0 missing
  [PASS] All records have tra

In [0]:
# Cell 2 — Gold layer checks
from pyspark.sql.functions import col

SILVER_TABLE = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE   = "aml_pipeline.transactions.gold_sar_reports"

gold = spark.table(GOLD_TABLE)
gold_count = gold.count()
silver_flagged = spark.table(SILVER_TABLE).filter("is_flagged = true").count()

passed = 0
failed = 0

def check(name, condition, detail=""):
    global passed, failed
    status = "PASS" if condition else "FAIL"
    if condition:
        passed += 1
    else:
        failed += 1
    print(f"  [{status}] {name} — {detail}")

print("GOLD LAYER CHECKS")
print("-" * 50)

check("Gold count matches Silver flagged", gold_count == silver_flagged,
      f"Gold={gold_count:,}, Flagged={silver_flagged:,}")

null_sar = gold.filter(col("sar_reference").isNull()).count()
check("All SARs have reference number", null_sar == 0, f"{null_sar} missing")

null_sev = gold.filter(col("sar_severity").isNull()).count()
check("All SARs have severity level", null_sev == 0, f"{null_sev} missing")

non_pending = gold.filter(col("report_status") != "PENDING_REVIEW").count()
check("All SARs are PENDING_REVIEW", non_pending == 0, f"{non_pending} not pending")

distinct_sars = gold.select("sar_reference").distinct().count()
check("No duplicate SAR references", distinct_sars == gold_count,
      f"{gold_count - distinct_sars} duplicates")

print(f"\n{'='*50}")
print(f"  DATA QUALITY REPORT — GOLD LAYER")
print(f"{'='*50}")
print(f"  Checks passed : {passed}")
print(f"  Checks failed : {failed}")
print(f"  Status        : {'ALL PASSED' if failed == 0 else 'FAILED'}")
print(f"\n  COMBINED TOTAL: 18 Bronze+Silver + {passed+failed} Gold = {18+passed+failed} checks")
print(f"  ALL CHECKS PASSED" if failed == 0 else "  INVESTIGATE FAILURES")
print(f"{'='*50}")

GOLD LAYER CHECKS
--------------------------------------------------
  [PASS] Gold count matches Silver flagged — Gold=3,356, Flagged=3,356
  [PASS] All SARs have reference number — 0 missing
  [PASS] All SARs have severity level — 0 missing
  [PASS] All SARs are PENDING_REVIEW — 0 not pending
  [PASS] No duplicate SAR references — 0 duplicates

  DATA QUALITY REPORT — GOLD LAYER
  Checks passed : 5
  Checks failed : 0
  Status        : ALL PASSED

  COMBINED TOTAL: 18 Bronze+Silver + 5 Gold = 23 checks
  ALL CHECKS PASSED
